In [3]:
from __future__ import annotations

import logging
from pathlib import Path
from typing import Tuple

import joblib
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB

DATASET_PATH = Path(r"D:\Python-Course\ML-Course\test\Machine-Learning-Algorithm\Gaussian-Naive-Bayes\Data\diabetes.csv")
MODEL_PATH = Path(
    r"D:\Python-Course\ML-Course\test\Machine-Learning-Algorithm\Gaussian-Naive-Bayes\gaussian_bayes.pkl"
)

TARGET_COLUMN = "Outcome"

TEST_SIZE = 0.20
RANDOM_STATE = 42

IMPUTE_COLUMNS = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
]

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)

logger = logging.getLogger(__name__)

def load_dataset(filepath: Path) -> pd.DataFrame:
    if not filepath.exists():
        raise FileNotFoundError(f"Dataset not found: {filepath}")

    logger.info("Loading dataset from %s", filepath)
    return pd.read_csv(filepath)

def split_dataset(
    data: pd.DataFrame,
    target_column: str,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:

    if target_column not in data.columns:
        raise ValueError(f"Target column '{target_column}' not found.")

    data[IMPUTE_COLUMNS] = data[IMPUTE_COLUMNS].replace(0, np.nan)

    imputer = SimpleImputer(strategy="median")
    data[IMPUTE_COLUMNS] = imputer.fit_transform(data[IMPUTE_COLUMNS])

    X = data.drop(columns=[target_column])
    y = data[target_column]

    return train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)

def build_model() -> GaussianNB:
    return GaussianNB()

def evaluate_model(model: GaussianNB, X_test: pd.DataFrame, y_test: pd.Series,
) -> None:
    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    logger.info("Model Accuracy : %.4f", accuracy)
    print("\nAccuracy")
    print("-" * 30)
    print(f"{accuracy:.4f}")

    print("\nConfusion Matrix")
    print("-" * 30)
    print(confusion_matrix(y_test, predictions))

    print("\nClassification Report")
    print("-" * 30)
    print(classification_report(y_test, predictions))

def save_model(model: GaussianNB, filepath: Path) -> None:
    joblib.dump(model, filepath)
    logger.info("Model saved to %s", filepath)

def main() -> None:
    try:
        data = load_dataset(DATASET_PATH)

        X_train, X_test, y_train, y_test = split_dataset(
            data,
            TARGET_COLUMN,
        )

        model = build_model()

        logger.info("Training model...")

        model.fit(X_train, y_train)

        logger.info("Training completed.")

        evaluate_model(
            model,
            X_test,
            y_test,
        )

        save_model(model, MODEL_PATH)

    except Exception:
        logger.exception("Application failed.")
        raise


if __name__ == "__main__":
    main()

2026-07-03 12:10:46,997 | INFO | Loading dataset from D:\Python-Course\ML-Course\test\Machine-Learning-Algorithm\Gaussian-Naive-Bayes\Data\diabetes.csv
2026-07-03 12:10:47,031 | INFO | Training model...
2026-07-03 12:10:47,042 | INFO | Training completed.
2026-07-03 12:10:47,053 | INFO | Model Accuracy : 0.7013
2026-07-03 12:10:47,096 | INFO | Model saved to D:\Python-Course\ML-Course\test\Machine-Learning-Algorithm\Gaussian-Naive-Bayes\gaussian_bayes.pkl



Accuracy
------------------------------
0.7013

Confusion Matrix
------------------------------
[[74 26]
 [20 34]]

Classification Report
------------------------------
              precision    recall  f1-score   support

           0       0.79      0.74      0.76       100
           1       0.57      0.63      0.60        54

    accuracy                           0.70       154
   macro avg       0.68      0.68      0.68       154
weighted avg       0.71      0.70      0.70       154

